# 🌳 Cuaderno 02 — Newton y Lagrange

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

De la tabla de diferencias divididas a la forma general de Newton, y de ahí a la fórmula de Lagrange. Dos formas del **mismo** polinomio.

> 📖 Teoría: [Newton](../docs/metodos/02_newton_diferencias_divididas.md) · [Lagrange](../docs/metodos/03_lagrange.md)

In [ ]:
#@title ⚙️ Entorno (ejecuta PRIMERO)
import os, sys, importlib

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _buscar(nombre, saltos=6):
    """Busca un archivo junto al cuaderno, en la carpeta actual y en lib/ de los padres."""
    bases = [os.getcwd()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    if ruta_nb:
        bases.append(os.path.dirname(os.path.abspath(ruta_nb)))
    for base in list(bases):
        b = base
        for _ in range(saltos):
            for cand in (os.path.join(b, nombre), os.path.join(b, "lib", nombre)):
                if os.path.exists(cand):
                    return os.path.abspath(cand)
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


rutas = {n: _buscar(n) for n in ("robomat.py", "ajuste_curvas.py")}
faltan = [n for n, r in rutas.items() if r is None]
if faltan:
    raise FileNotFoundError(
        "No encontré " + ", ".join(faltan) +
        ". En Colab, sube la carpeta lib/ junto al cuaderno.")

_carpeta_lib = os.path.dirname(rutas["robomat.py"])
if _carpeta_lib not in sys.path:
    sys.path.insert(0, _carpeta_lib)

import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

import ajuste_curvas as ac
importlib.reload(ac)

import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider
except ImportError:
    interact = None

print("✅ Entorno cargado desde:", _carpeta_lib)
print("✅ Entorno:", "Google Colab" if ES_COLAB else "local")

### 🎯 Objetivos

- Construir la tabla de diferencias divididas y leer los coeficientes en su diagonal.
- Evaluar el polinomio de Newton con Horner y comparar grados sucesivos.
- Evaluar el polinomio de Lagrange y comprobar que es el mismo polinomio.
- Estimar el error con la ecuación (18.18).

**Referencia del libro:** secciones 18.1 y 18.2, ejemplos 18.3, 18.4 y 18.6.

## 1. La tabla de diferencias divididas

Con los nodos $x=[1,4,6,5]$ y $f=[0,1.386294,1.791759,1.609438]$, la tabla se
construye por columnas:

$$f[x_i,x_j]=\frac{f(x_i)-f(x_j)}{x_i-x_j}, \tag{18.12}$$

$$f[x_i,x_j,x_k]=\frac{f[x_i,x_j]-f[x_j,x_k]}{x_i-x_k}. \tag{18.13}$$

Los coeficientes $b_0,\dots,b_n$ del polinomio de Newton son la **diagonal**.

In [ ]:
x = np.array([1.0, 4.0, 6.0, 5.0])
f = np.array([0.0, 1.386294, 1.791759, 1.609438])
T = ac.tabla_diferencias(x, f)

print("Tabla (columnas = orden de la diferencia):")
for i, fila in enumerate(T):
    print(f"  x={x[i]:>4.1f} | " + " ".join(f"{v:>12.7f}" for v in fila[: i + 1]))
print("\ncoeficientes b =", np.diag(T))

**Comprobación manual.** La primera diferencia es
$f[x_1,x_0]=(1.386294-0)/(4-1)=0.4620981$; la segunda es
$f[x_2,x_1,x_0]=(0.2027326-0.4620981)/(6-1)=-0.0518731$; la tercera es
$0.007865529$. Coinciden con la diagonal.

## 2. Forma general y evaluación

$$f_n(x)=b_0+b_1(x-x_0)+\cdots+b_n(x-x_0)\cdots(x-x_{n-1}). \tag{18.7}$$

La evaluación se hace con Horner, de adentro hacia afuera:

$$p=b_n;\qquad p\leftarrow p(x-x_k)+b_k,\quad k=n-1,\dots,0.$$

In [ ]:
b = ac.coeficientes_newton(x, f)

for grado in range(1, 5):
    xg, fg = x[:grado+1], f[:grado+1]
    pred = ac.newton(xg, fg, 2.0)
    err = abs(pred - 0.6931472) / 0.6931472 * 100
    print(f"grado {grado}: f(2) = {pred:.7f}   error = {err:6.2f}%")

In [ ]:
xs = np.linspace(0.8, 6.4, 400)
plt.figure(figsize=(9, 5))
plt.plot(xs, np.log(xs), "k", lw=2, label="ln x")
for grado, estilo in [(1, ":"), (2, "--"), (3, "-."), (4, "-")]:
    plt.plot(xs, ac.newton(x[:grado+1], f[:grado+1], xs), estilo,
             label=f"Newton grado {grado}")
plt.plot(x, f, "o", ms=8, label="nodos")
plt.ylim(-0.5, 2.2); plt.xlabel("x"); plt.ylabel("f(x)")
plt.legend(); plt.grid(alpha=0.3)
plt.title("El grado alto no siempre es mejor")
plt.show()

## 3. Lagrange: el mismo polinomio, otra forma

$$f_n(x)=\sum_{i=0}^{n}L_i(x)f(x_i),\qquad
L_i(x)=\prod_{\substack{j=0\\j\neq i}}^{n}\frac{x-x_j}{x_i-x_j}. \tag{18.20–18.21}$$

Cada $L_i$ vale 1 en su nodo y 0 en los demás. Por eso la suma pasa
exactamente por todos los puntos.

In [ ]:
for grado in range(1, 5):
    xg, fg = x[:grado+1], f[:grado+1]
    lag = ac.evaluar_lagrange(xg, fg, 2.0)
    new = ac.newton(xg, fg, 2.0)
    print(f"grado {grado}: Lagrange={lag:.10f}  Newton={new:.10f}  "
          f"diferencia={abs(lag-new):.2e}")

In [ ]:
# Los pesos L_i(x) suman 1 para cualquier x: compruébalo.
xp = 2.0
pesos = []
for i in range(len(x)):
    Li = 1.0
    for j in range(len(x)):
        if i != j:
            Li *= (xp - x[j]) / (x[i] - x[j])
    pesos.append(Li)
print("pesos =", np.round(pesos, 6), " suma =", sum(pesos))
print("suma ponderada =", sum(Li * fi for Li, fi in zip(pesos, f)))

## 4. Estimación del error

Con la cuadrática y el punto extra $x_3=5$:

$$R_2=f[x_3,x_2,x_1,x_0](x-x_0)(x-x_1)(x-x_2). \tag{18.18}$$

El libro obtiene $0.0629242$, del mismo orden que el error verdadero
$0.1273028$.

In [ ]:
R2 = T[3, 3] * (2.0 - x[0]) * (2.0 - x[1]) * (2.0 - x[2])
f2 = ac.newton(x[:3], f[:3], 2.0)
print(f"f2(2) = {f2:.7f}")
print(f"R2 estimado = {R2:.7f}")
print(f"error verdadero = {abs(f2 - 0.6931472):.7f}")
print(f"diferencia entre grados 3 y 2 = {ac.newton(x[:4], f[:4], 2.0) - f2:.7f}")

## 5. Autoevaluación

**Pregunta 1.** ¿Cuánto vale $f[x_2,x_1]$ con $x_1=4$, $f_1=1.386294$ y
$x_2=6$, $f_2=1.791759$?

In [ ]:
respuesta_1 = 0.0     # <-- cambia este valor

correcta_1 = (1.791759 - 1.386294) / (6 - 4)
print("✅ Correcto" if verificar(respuesta_1, correcta_1, tol=1e-4)
      else f"❌ Revisa la ecuación (18.12). Valor esperado: {correcta_1:.7f}")

**Pregunta 2.** Evalúa con Lagrange, en $x=2$, la recta que pasa por
$(1,0)$ y $(4,1.386294)$.

In [ ]:
respuesta_2 = 0.0     # <-- cambia este valor

correcta_2 = ac.evaluar_lagrange([1.0, 4.0], [0.0, 1.386294], 2.0)
print("✅ Correcto" if verificar(respuesta_2, correcta_2, tol=1e-4)
      else f"❌ Revisa la ecuación (18.22). Valor esperado: {correcta_2:.7f}")

**Pregunta 3 (decisión).** Tienes que interpolar 12 veces con los mismos 8
nodos. ¿Newton o Lagrange? Escribe la justificación.

In [ ]:
respuesta_3 = '''
Escribe aquí tu justificación (piensa en el costo de recalcular).
'''
print(respuesta_3)

## 6. Resumen

| | Newton | Lagrange |
|---|---|---|
| Cálculo previo | tabla $O(n^2)$ | ninguno |
| Agregar nodo | un término más | recalcular todo |
| Error estimado | sí, con (18.18) | no en la práctica |
| Programación | dos ciclos | dos ciclos |

Ambos producen **el mismo polinomio**. La elección es de eficiencia y de
necesidad de estimar el error, no de exactitud.